# Background

Shazam works by using *audio fingerprinting*: given a song, it generates a set of identifiers, and searches an audio database to find a match and identify the song. I used my own code based on lessons from my favorite concept: DTFS in EECS 16A!

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.signal as signal
import pandas as pd
import IPython.display as ipd

from scipy.io import wavfile
from scipy.ndimage import maximum_filter
from shazam_utils import hashing

## Glossary

CT: Continuous Time - time progresses continuously

DT: Discrete Time - time is undefined for all other real numbers within some interval

FT: Fourier Transform - a mathematical operation to convert time signal or function into frequencies

DFT: Discrete Fourier Transform - a Fourier transform that takes in a discrete time-domain signal and outputs a discrete frequency-domain signal within the magnitude of each frequency

DTFT: Discrete Time Fourier Transform - a Fourier transform that takes in a discrete time-domain signal and outputs a continuous frequency-domain signal

In [ ]:
fs, coldplay = wavfile.read("VivaLaVida.wav")
print(f"Audio Shape: {coldplay.shape}, Sampling Rate: {fs} Hz")
coldplay = np.mean(coldplay, axis=1)

In [ ]:
plt.figure(figsize=(16, 4), dpi=200)
coldplay_labels = [x / fs for x in range(fs*10)]
plt.plot(coldplay_labels, coldplay[:fs*10])
plt.xlabel("Time [s]")
plt.ylabel("Magnitude")
plt.title("Time-domain visualization of first 10 seconds of Viva La Vida")
plt.show()

In [ ]:
ipd.Audio("VivaLaVida.wav")

## Centered Magnitude Spectrum

As far as spectral analysis is concerned, it seems like we should just be able to take the DFT of the entire song, find our fingerprints, and be done, right? Nope! We have to:

1. Perform a 1-D DFT on `sig`
2. Shift the DFT to $[-\pi$, $\pi]$
3. Find the magnitude of the shifted DFT, i.e. $|X(\omega)|$

In [ ]:
def centered_magnitude_spectrum(sig):
    """
    Inputs:
    sig - a generic iterable signal of floating point numbers
    
    Output (np.ndarray):
    Returns a centered magnitude spectrum of the given signal. 
    That is, the magnitude of the DFT of the provided signal 
    after shifting from [0,2pi] to [-pi,pi].
    """
    
    # TODO YOUR CODE HERE
    #1. Compute the 1-D DFT
    X = np.fft.fft(sig)

    #2. Shift zero-frequency component to center (-pi to pi)
    X_shifted = np.fft.fftshift(X)

    #3. Take magnitude
    magnitude = np.abs(X_shifted)

    return magnitude

    ...

To see why one DFT won't suffice, we're going to look at the spectrum of different sections of Viva La Vida.

First, we'll look at magnitude spectrum of the first 10 seconds of the song.

In [ ]:
coldplay_cropped = coldplay[:10 * fs]
coldplay_freqs = centered_magnitude_spectrum(coldplay_cropped)

plt.figure(figsize=(16, 4), dpi=200)
freqs = np.linspace(-fs/2, fs/2, len(coldplay_freqs))
plt.plot(freqs, coldplay_freqs)
plt.xlabel("Frequency [Hz]")
plt.ylabel("Magnitude")
plt.title("DFT of first 10 seconds of Viva La Vida")
plt.show()

In [ ]:
coldplay_freqs_1 = centered_magnitude_spectrum(coldplay[:fs]) 
coldplay_freqs_2 = centered_magnitude_spectrum(coldplay[fs:2*fs])
coldplay_freqs_3 = centered_magnitude_spectrum(coldplay[2*fs:3*fs])
coldplay_freqs_4 = centered_magnitude_spectrum(coldplay[3*fs:4*fs])

freqs = np.linspace(-fs / 2, fs / 2, len(coldplay_freqs_1))
sigs = [coldplay_freqs_1, coldplay_freqs_2, coldplay_freqs_3, coldplay_freqs_4]
strs = ["1st", "2nd", "3rd", "4th"]

plt.figure(figsize=(16, 10), dpi=200)
for i in range(1, 5):
    plt.subplot(2, 2, i)
    plt.plot(freqs, sigs[i-1])
    plt.xlim([-.5e4, .5e4])
    plt.ylim([0, 1.1 * np.array(sigs).max()])
    plt.title("DFT magnitude of {} second of Viva La Vida".format(strs[i-1]))
plt.show()

## Spectrogrammin'

A *spectrogram* is an image representing the frequency content of a signal at different times. For visual context, see the speech spectrogram below.

![speech-spectrogram.png](speech-spectrogram.png)

In the cell below, we generate 1000 samples of the following signals for one second (i.e., over the interval $t \in [0, 1]$):
- A 100 Hz sine wave (call it `x1`).
- A 400 Hz sine wave (call it `x2`).
- A third signal, call it `x3`, by concatenating `x1` and `x2`.

In [ ]:
n = np.linspace(0, 1, 1000)
x1 = np.sin(2 * np.pi * 100 * n)
x2 = np.sin(2 * np.pi * 400 * n)
x3 = np.concatenate((x1, x2))

First, let's look at the DFT of our signals.

In [ ]:
freqs_1 = centered_magnitude_spectrum(x1)
freqs_2 = centered_magnitude_spectrum(x2)
freqs_3 = centered_magnitude_spectrum(x3)

fig, axs = plt.subplots(1, 3, figsize=(16, 4), dpi=200)

axs[0].plot(np.linspace(-500,500, len(freqs_1)), freqs_1)
axs[0].set_ylabel('DFT Magnitude')
axs[0].set_xlabel('Frequency [Hz]')
axs[0].set_title("100 Hz Signal")

axs[1].plot(np.linspace(-500,500, len(freqs_2)), freqs_2)
axs[1].set_ylabel('DFT Magnitude')
axs[1].set_xlabel('Frequency [Hz]')
axs[1].set_title("400 Hz Signal")

axs[2].plot(np.linspace(-500,500, len(freqs_3)), freqs_3)
axs[2].set_ylabel('DFT Magnitude')
axs[2].set_xlabel('Frequency [Hz]')
axs[2].set_title("100 Hz and 400 Hz Signal")
plt.show()

As expected, the *pure tones* (the 100 Hz and 400 Hz sine waves) have 2 peaks each, whereas the signal formed by concatenating them has 4 peaks.

In [ ]:
f1, t1, x1_freqs = signal.spectrogram(x1, fs=1000)
f2, t2, x2_freqs = signal.spectrogram(x2, fs=1000)
f3, t3, x3_freqs = signal.spectrogram(x3, fs=1000)

fig, axs = plt.subplots(1, 3, figsize=(16, 4), dpi=200)

axs[0].pcolormesh(t1, f1, x1_freqs, cmap="gray", shading='auto')
axs[0].set_ylabel('Frequency [Hz]')
axs[0].set_xlabel('Time [sec]')
axs[0].set_title("100 Hz Signal")

axs[1].pcolormesh(t2, f2, x2_freqs, cmap="gray", shading='auto')
axs[1].set_ylabel('Frequency [Hz]')
axs[1].set_xlabel('Time [sec]')
axs[1].set_title("400 Hz Signal")

axs[2].pcolormesh(t3, f3, x3_freqs, cmap="gray", shading='auto')
axs[2].set_ylabel('Frequency [Hz]')
axs[2].set_xlabel('Time [sec]')
axs[2].set_title("100 Hz and 400 Hz Signal")

plt.show()

In [ ]:
fs, coldplay = wavfile.read("VivaLaVida.wav")
coldplay = np.mean(coldplay, axis=1)

fs, killers = wavfile.read("MrBrightside.wav")
killers = np.mean(killers, axis=1)

In [ ]:
ipd.Audio("MrBrightside.wav")

In [ ]:
def compute_spectrogram(fs, audio, epsilon_db_constant):
    """
    Input:
    fs - the sampling frequency of the audio, in Hertz (Hz)
    audio - the full audio to compute the spectrogram of; either coldplay or killers
    epsilon_db_constant - a small positive constant to ensure there are no divide by zero errors
    
    Output (np.ndarray, np.ndarray, np.ndarray):
    Returns a scipy spectrogram for the given audio (in decibels) with three components:
     - a NumPy array of sample frequencies
     - a NumPy array of segment times
     - the spectrogram itself
    
    See:
    scipy.signal.spectrogram
    numpy.log10
    """
    
    # TODO YOUR CODE HERE
    #1. Compute spectrogram with 4096-point DFT
    freqs, times, Sxx = signal.spectrogram(audio, fs=fs, nperseg=4096)

    #2. Convert power spectrogram to decibel
    Sxx_db = 20 * np.log10(Sxx + epsilon_db_constant)

    return freqs, times, Sxx_db

    ...

Let's have a look! Run the cell below to compute spectrograms for both *Viva La Vida* and *Mr. Brightside*, then plot their spectrograms.

In [ ]:
# Compute spectrogram for Viva La Vida
f1, t1, coldplay_spect = compute_spectrogram(fs, coldplay, epsilon_db_constant=1e-12)

# Compute spectrogram for Mr. Brightside
f2, t2, killers_spect = compute_spectrogram(fs, killers, epsilon_db_constant=1e-12)

plt.figure(figsize=(20, 10), dpi=200)

plt.subplot(2, 1, 1)
plt.pcolormesh(t1, f1, coldplay_spect, cmap="jet", shading="auto")
plt.ylabel("Frequency [Hz]")
plt.xlabel("Time [sec]")
plt.title("Viva La Vida")
plt.colorbar()

plt.subplot(2, 1, 2)
plt.pcolormesh(t2, f2, killers_spect, cmap="jet", shading="auto")
plt.ylabel("Frequency [Hz]")
plt.xlabel("Time [sec]")
plt.title("Mr. Brightside")
plt.colorbar()

plt.show()

> **Answer (spectrogram analysis):** Dark blue means no signal power. At the start of both songs, there is silence (quiet sound). When we take DFT of that chunk of the DT signal, there is almost no energy at any frequency, so spectrum is almost zero. When we convert to decibels, it is a very small amount as negative numbers like -300 dB. The two peaks extending towards 20kHz in Mr. Brightside indicate high frequencies. At the beginning of Mr. Brightside, you hear sharp drum sounds. Those peaks are hi-hat cymbals, which are high-frequency. Thus, they are high in peak in the spectrogram. You can tell songs apart by strength of frequencies, when they happen, and how long they last. A single DFT would have information on these, but spectrogram shows frequency vs. time.

# Getting to Fingerprinting

Our end goal here is to take an audio snippet and figure out what song's being played. 

When our version of Shazam gets fed a song to classify, it can compare small snippets of the fingerprints, rather than looking at the whole song, which would resolve our storage issues with OCF computers.

1. ***Temporal Locality and Translational Invariance:*** We're trying to figure out what song is being played based on a short (say, 5 to 10 second long) clip. So, our fingerprints should somehow encode *where* in the song they come from.

2. ***Robustness:*** An audio file, whether clean or degraded by (a modest amount of) noise, should produce the same fingerprint.

3. ***High Entropy:*** The fingerprinting algorithm should be "random enough" to have unique footprints.

**So, spectrograms are cool, but how can we use them? They contain thousands of points... how do we pick which are the most important?**

As you might guess, we'll look at the spectrogram's *peaks*: points in high-energy areas, which are mostly clear from all that noise.

$$g(x, y) = \max_{i,j} f(x+i, y+j) \text{  where } -25\le i, j \le 25.$$

1. Implement the maximum filter and apply it to the provided spectrogram. When the neighborhood exceeds the boundary of the image, assume $f(x, y)$ is the value of the image at that point (i.e., set `mode='constant'` and `size=neighborhood_size`).
2. Extract a boolean mask which is True when $f(x, y) = g(x, y)$, and False otherwise.
3. To ensure these peaks are big enough, in the mask, set any peak locations with a peak less than or equal to `AMP_THRESH` to zero. This is filled in for you.

In [ ]:
def peak_finding(spect, neighborhood_size=2*25+1, amp_thresh=40):
    """
    Input:
    spect - the spectrogram of an unknown audio track to find peaks from
    neighborhood_size - the size of the maximum filter
    amp_thresh - amplitude threshold to include peaks in result
    
    Output (np.ndarray, np.ndarray):
    Returns a tuple of the peak indices on the frequency 
    and time axes (each as NumPy arrays) for the provided spectrograph.
    
    See:
    maximum_filter
    np.nonzero
    """


    # Apply a Maximum Filter
    max_spect = maximum_filter(spect, size=neighborhood_size, mode='constant')

    # Compute the mask
    mask = (spect == max_spect)

    # Filter out tiny peaks
    mask &= spect > amp_thresh

    # Get the indices of the peaks
    freq_indices, time_indices = np.nonzero(mask)

    return freq_indices, time_indices

# Call peak_finding with the spectrogram for Viva La Vida
freq_indices, time_indices = peak_finding(coldplay_spect) 

In [ ]:
plt.figure(figsize=(16, 6), dpi=200)
plt.scatter(t1[time_indices], f1[freq_indices], zorder=99, color='k')
plt.pcolormesh(t1, f1, coldplay_spect, zorder=0, cmap="jet", shading="auto")
plt.ylabel('Frequency [Hz]')
plt.xlabel('Time [sec]')
plt.title("Spectrogram Peaks (for Viva La Vida)")
plt.xlim([0, 60])
plt.show()

In [ ]:
def fingerprint(fs, audio, neighborhood_size=2*25+1, amp_thresh=40):
    """
    Input:
    fs - the sampling frequency of the audio, in Hertz (Hz)
    audio - the full audio to fingerprint; either coldplay or killers
    neighborhood_size - the size of the maximum filter
    amp_thresh - amplitude threshold to include peaks in result
    
    Output (list[str, int]):
    A list of hashes representing the "fingerprint" of the given audio.
    """
    
    audio = np.mean(audio, axis=1)
    
    # Compute the spectrogram of the single channel audio
    f1, t1, spect = signal.spectrogram(audio, fs=fs, nperseg=4096)

    # Find the peaks (Use function from Q2a)
    freq_indices, time_indices = peak_finding(spect, neighborhood_size=neighborhood_size, amp_thresh=amp_thresh)
    
    # Compute the hashes
    hashes = hashing(f1, t1, freq_indices, time_indices)
    
    # Return list of hashes
    return hashes

In [ ]:
def detect(fs, audio):
    db = pd.read_csv("database.csv", header=None, names=["Hash", "time", "Song"])

    hashes = fingerprint(fs, audio)
    db_matches = db[db.Hash.isin(map(lambda x: x[0], hashes))]
    if len(db_matches) == 0:
        print("No Matches")
        return

    counts = db_matches.groupby("Song").size()
    counts = counts / counts.sum()
    return counts.idxmax(), counts.max() * 100

In [ ]:
def get_20_second_segment(fs, audio):
    """
    Input:
    fs - the sampling frequency of the audio, in Hertz (Hz)
    audio - the full audio to get 20 seconds of; either coldplay or killers
    
    Output:
    A 20 second segment anywhere within the given audio track.
    
    Example:
    get_20_second_segment(killers) == killers[X seconds:(X + 20 seconds)]
    """
    

    total_seconds = len(audio) // fs #without remainder
    start_second = np.random.randint(0, total_seconds - 20)

    start_sample = start_second * fs
    end_sample = (start_second + 20) * fs

    return audio[start_sample:end_sample]

You will now use this function to write tests for your Shazam system!

In [ ]:
fs, coldplay = wavfile.read("VivaLaVida.wav")
fs, killers = wavfile.read("MrBrightside.wav")

In [ ]:
def basic_detect_test(fs, audio):
    """
    Input:
    fs - the sampling frequency of the audio, in Hertz (Hz)
    audio - the full audio to detect against; either coldplay or killers
    
    Output:
    Returns the name of the audio track that most closely matches 
    a 20 second segment of the provided audio track, and a percentage confidence.
    
    Example:
    basic_detect_test(killers_fs, killers) == ('MrBrightside.wav', 100.0)
    
    See also:
    get_20_second_segment
    detect
    """
    
    segment = get_20_second_segment(fs, audio)

    return detect(fs, segment)
    
    

In [ ]:
NOISE_MEAN = 10000
NOISE_STANDARD_DEVIATION = 10000

def add_gaussian_noise(audio_segment):
    """
    Input:
    audio_segment - an audio segment from an unknown track
    
    Output:
    Returns the audio segment with added Gaussian noise.
    
    See:
    Problem description (for quantities)
    np.random.normal
    """
    
    # TODO YOUR CODE HERE

    noise = np.random.normal(NOISE_MEAN, NOISE_STANDARD_DEVIATION, size=audio_segment.shape)
    return audio_segment + noise

In [ ]:
def gaussian_noise_detect_test(fs, audio_segment):
    """
    Input:
    fs - the sampling frequency of the audio, in Hertz (Hz)
    audio_segment - an audio segment from an unknown track WITHOUT Gaussian noise
    
    Output:
    Returns the name of the audio track that most closely matches 
    a 20 second segment of the provided audio track, WITH added 
    Gaussian noise and a percentage confidence.
    
    See:
    add_gaussian_noise
    detect
    """
    
    # TODO YOUR CODE HERE
    noisy_segment = add_gaussian_noise(audio_segment)

    return detect(fs, noisy_segment)

    ...

In [ ]:
ipd.Audio(add_gaussian_noise(get_20_second_segment(fs, coldplay)).T, rate=fs)

In [ ]:
grader.check("q3c")

In [ ]:
unknown_segment = coldplay[10 * fs: 30 * fs].copy()
unknown_segment[:2 * fs] = 0
unknown_segment[6 * fs:8 * fs] = 0
unknown_segment[16 * fs:20 * fs] = 0
unknown_segment[2 * fs:4 * fs] = 0

In [ ]:
ipd.Audio(unknown_segment.T, rate=fs)

In [ ]:
detect(fs, unknown_segment)

In [ ]:
unknown_segment = killers[10 * fs: 30 * fs].copy()
unknown_segment[:2 * fs] = 0
unknown_segment[6 * fs:8 * fs] = 0
unknown_segment[16 * fs:20 * fs] = 0
unknown_segment[2 * fs:4 * fs] = 0

In [ ]:
ipd.Audio(unknown_segment.T, rate=fs)

In [ ]:
detect(fs, unknown_segment)

In [ ]:
import csv
def add_to_db(filename):
    fs, audio = wavfile.read(filename)
    hashes = fingerprint(audio, fs)
    with open('database.csv', mode='a') as db_file:
        db_writer = csv.writer(db_file, delimiter=',', quotechar='"', quoting=csv.QUOTE_MINIMAL)

        for hash_pair in hashes:
            db_writer.writerow([hash_pair[0], hash_pair[1], filename])

In [ ]:
my_wav_filepath = diditagain_mp3  # Path to any WAV file you want to add to the database
add_to_db(my_wav_filepath)

Then run this to detect against your updated database. Inputting the same WAV file should result in your new WAV file being detected! Try with a WAV file that isn't in the database, or a WAV file that is a composite of multiple songs in the database. Experiment and see what happens!

In [ ]:
fs, audio = wavfile.read(___)  # Path to any WAV file to detect against
detect(fs, audio)

## Conceptual Notes

* A computer stores in numbers in memory only working with finite arrays, so cannot handle continuous integrals in true audio samples. Through DFT formula, summation over a number samples. When we set up Fourier transform using vectors, it is a dot product between your signal vector and frequency vector (projection of signal onto frequency basis vector)

* Audio signals are not continuous. In real life, they are continuous, but computers cannot store continuous signals. We sample it at a fixed rate (like 48 kHz), so we have a DT signal at discrete steps: n = 0, 1, 2, 3

  
* Fourier Transform (FT) assumes x(t) is defined for all real numbers. We only have samples and finite time length, so we use DFT.

* The complex exponentials form an orthogonal basis spanning length-N signals. DFT is not random, it is a change of basis time domain to frequency domain basis. Because vectors are orthogonal, computing dot product with one frequency is possible as other frequencies cancel out (good for isolating each frequency)

* Spectrograms provide temporal locality (where in the song) stored as `(hash, time_index)`. Compare time shifts, spectrogram same time shift → strong match (spectrogram keeps time and so do peaks)
* DFT: tells us what frequencies are in entire signal, but loses time info
* Music is a temporal signal when do drums enter? when do vocals begin? when do chords change? For this, we use a spectrogram.

What steps are we following?

* Split signals into short chunks (filtering / windowing)
* Compute DFT of each chunk
* Stack results side by side

* What does fingerprinting do? After finding the peaks, we have frequency vs. time (t, f) known as a constellation map.
* Instead of storing and analyzing the entire song at once, we store the relationships between peaks. We link it as (t(i+1), f(i+1)) and compute hash using f1, f2, time difference (f1, f2, delta t) → Compute many DFTs (frequency map)
* Suppose there are snippets from beginning, middle, and end of a song, the absolute time changes, but time difference between peaks stay the same/constant
* Peak finding: strong local maxima and values above threshold → constellation map
* Hashes: local structure, time spacing, frequency values → two diff songs cannot produce same fingerprints
Even small musical differences by hearing differ in peak locations, time values, and hash values with probability of collision being low -> high entropy
constant being zero handles outside values
* Matching: song with most matches
* Gaussian noise: adds energy everywhere, low amplitude, random distribution
* Noise: not creating structured peaks or consistent peak pairs so original peaks survive → robustness